### DecisionTree

In [1]:
import os
import pickle
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score, roc_auc_score, precision_recall_fscore_support
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from hmmlearn import hmm
from hmmlearn.hmm import GaussianHMM
import plotly.express as px
import mlflow
from mlflow.models.signature import infer_signature
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import TomekLinks
from imblearn.combine import SMOTETomek

In [2]:
files = ['two_class_raw_1s_no.csv', 'two_class_raw_1s_yo_0.5.csv', 'two_class_raw_1s_yo_0.8.csv', 'two_class_raw_2s_no.csv', 'two_class_raw_2s_yo_0.5.csv', 'two_class_raw_2s_yo_0.8.csv', 
        'two_class_raw_3s_no.csv', 'two_class_raw_3s_yo_0.5.csv', 'two_class_raw_3s_yo_0.8.csv', 'two_class_raw_4s_no.csv', 'two_class_raw_4s_yo_0.5.csv', 'two_class_raw_4s_yo_0.8.csv',
        'two_class_raw_5s_no.csv', 'two_class_raw_5s_yo_0.5.csv', 'two_class_raw_5s_yo_0.8.csv']

base_path = '/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/improved_features_data_main/'

In [3]:
def compare_sampling_techniques(X_train, y_train, groups_train, random_state=42):
    """
    Compare different sampling techniques for imbalanced classification
    """
    
    # Define sampling strategies to test
    sampling_strategies = {
        'none': None,
        'oversample_smote': SMOTE(random_state=random_state),
        'undersample_tomek_links': TomekLinks(),
        'combined_smote_tomek': SMOTETomek(random_state=random_state)
    }
    
    results = {}
    
    for strategy_name, sampler in sampling_strategies.items():
        print(f"\nTesting {strategy_name}...")
        
        # Cross-validation setup
        gkf = GroupKFold(n_splits=10)  # Using 5 folds for faster initial testing
        fold_results = []
        
        for fold_num, (train_idx_fold, test_idx_fold) in enumerate(gkf.split(X_train, y_train, groups_train)):
            X_train_fold, X_test_fold = X_train.iloc[train_idx_fold], X_train.iloc[test_idx_fold]
            y_train_fold, y_test_fold = y_train.iloc[train_idx_fold], y_train.iloc[test_idx_fold]

            
            # Apply sampling (if any)
            if sampler is not None:
                try:
                    X_train_resampled, y_train_resampled = sampler.fit_resample(X_train_fold, y_train_fold)
                    X_train_resampled = pd.DataFrame(X_train_resampled, columns=X_train_fold.columns)
                    y_train_resampled = pd.Series(y_train_resampled)
                except Exception as e:
                    print(f"Sampling failed for {strategy_name}: {e}")
                    continue
            else:
                X_train_resampled = X_train_fold
                y_train_resampled = y_train_fold
            
            # Use DecisionTreeClassifier with DEFAULT parameters
            model = DecisionTreeClassifier(
                random_state=random_state,
            )
            
            # Encode labels
            label_encoder = LabelEncoder()
            y_train_encoded = label_encoder.fit_transform(y_train_resampled)
            
            # Fit model
            model.fit(X_train_resampled, y_train_encoded)
            
            # Predict
            predictions = label_encoder.inverse_transform(model.predict(X_test_fold))
            
            # Calculate metrics
            accuracy = accuracy_score(y_test_fold, predictions)
            report_dict = classification_report(y_test_fold, predictions, output_dict=True)

            precision_void = report_dict.get("void", {}).get("precision", 0.0)
            recall_void = report_dict.get("void", {}).get("recall", 0.0)
            f1_void = report_dict.get("void", {}).get("f1-score", 0.0)

            precision_non_void = report_dict.get("non-void", {}).get("precision", 0.0)
            recall_non_void = report_dict.get("non-void", {}).get("recall", 0.0)
            f1_non_void = report_dict.get("non-void", {}).get("f1-score", 0.0)

            macro_f1 = report_dict.get("macro avg", {}).get("f1-score", 0.0)
            
            # Store detailed results
            fold_results.append({
                'fold': fold_num,
                'recall_void': recall_void,
                'precision_void': precision_void,
                'f1_void': f1_void,
                'macro_f1': macro_f1,
                'accuracy': accuracy,
                'precision_non_void': precision_non_void,  # Assuming 'non-void' is majority
                'recall_non_void': recall_non_void,
                'f1_non_void': f1_non_void,
                'class_distribution_train': dict(y_train_resampled.value_counts()),
                'class_distribution_test': dict(y_test_fold.value_counts())
            })
            

        # Calculate summary statistics
        if fold_results:  # Only if we have valid results
            results[strategy_name] = {
                'mean_accuracy': np.mean([f['accuracy'] for f in fold_results]),
                'std_accuracy': np.std([f['accuracy'] for f in fold_results]),
                'mean_recall_minority': np.mean([f['recall_void'] for f in fold_results]),
                'std_recall_minority': np.std([f['recall_void'] for f in fold_results]),
                'mean_f1_minority': np.mean([f['f1_void'] for f in fold_results]),
                'std_f1_minority': np.std([f['f1_void'] for f in fold_results]),
                'mean_f1_majority': np.mean([f['f1_non_void'] for f in fold_results]),
                'fold_details': fold_results
            }
    
    return results

In [4]:
file_results = {}
for file in tqdm(files, desc="Producing results for different sampling techniques"):
    data_path = os.path.join(base_path, file)
    features = pd.read_csv(data_path)
    features.drop(['center_time', 'start_time', 'end_time'], axis=1, inplace=True)
    details = file.split('_')
    exp_name = f"{details[3]}_{details[-1].replace('.csv', '')}"
    print(f"Analysing {exp_name}")
    
    # split data
    X = features.drop(columns=['label', 'experiment_id'])
    y = features['label']
    groups = features['experiment_id']

    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(splitter.split(X, y, groups))

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    groups_train, groups_test = groups.iloc[train_idx], groups.iloc[test_idx]
    
    file_results[exp_name] = compare_sampling_techniques(X_train, y_train, groups_train, 42)

Producing results for different sampling techniques:   0%|          | 0/15 [00:00<?, ?it/s]

Analysing 1s_no

Testing none...

Testing oversample_smote...

Testing undersample_tomek_links...

Testing combined_smote_tomek...


Producing results for different sampling techniques:   7%|▋         | 1/15 [00:11<02:47, 12.00s/it]

Analysing 1s_0.5

Testing none...

Testing oversample_smote...

Testing undersample_tomek_links...

Testing combined_smote_tomek...


Producing results for different sampling techniques:  13%|█▎        | 2/15 [00:40<04:38, 21.46s/it]

...

Producing results for different sampling techniques:  87%|████████▋ | 13/15 [03:44<00:16,  8.10s/it]

Analysing 5s_0.5

Testing none...

Testing oversample_smote...

Testing undersample_tomek_links...

Testing combined_smote_tomek...


Producing results for different sampling techniques:  93%|█████████▎| 14/15 [03:47<00:06,  6.69s/it]

Analysing 5s_0.8

Testing none...

Testing oversample_smote...

Testing undersample_tomek_links...

Testing combined_smote_tomek...


Producing results for different sampling techniques: 100%|██████████| 15/15 [03:57<00:00, 15.82s/it]


Pickle results.

We don't want any trouble charley

In [5]:
# pickle the file
with open('/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/imb_files/dt_imb_152.pkl', 'wb') as f:
    pickle.dump(file_results, f)